In [1]:
from utils import *
import wandb
from datetime import datetime

In [2]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [3]:
def saveExperiment(imageModel, textModel, config, experimentName, start):
    latestPath = os.path.join("checkpoints", "finetune", "latest")
    if not os.path.exists(os.path.join("checkpoints", "finetune", "latest")):
        os.mkdir(latestPath)

    stamp = start.strftime("%Y-%m-%d %H-%M")
    timePath = os.path.join("checkpoints", "finetune", stamp)
    if not os.path.exists(timePath):
        os.mkdir(timePath)

    saveToPath(latestPath, imageModel, textModel, config, experimentName)
    saveToPath(timePath, imageModel, textModel, config, experimentName)


def saveToPath(path, imageModel, textModel, config, experimentName):
    if not os.path.exists(os.path.join(path, experimentName)):
        os.mkdir(os.path.join(path, experimentName))

    torch.save(imageModel.state_dict(), os.path.join(path, experimentName, "image.pt"))
    # textModel.save_pretrained(os.path.join(path, experimentName, "text"))
    torch.save(textModel.state_dict(), os.path.join(path, experimentName, "text.pt"))
    config.save(os.path.join(path, experimentName, "config.json"))

In [4]:
def getTrainableParams(model):
    return [p for p in model.parameters() if p.requires_grad]

def setFrozen(model, frozen: bool):
    for param in model.parameters():
        param.requires_grad = not frozen

def unfreezeTopBlocks(imageModel, textModel: CLIPTextEmbedder, n: int):
    """
    Unfreeze the last n transformer blocks of both models and keep both heads trainable.
    Image model: self.transformer.layers
    Text model:  self.transformer.layers
    """

    if n == len(imageModel.model.transformer.layers):
        setFrozen(imageModel, False)
        setFrozen(textModel, False)
        return

    def freezeLayers(layers):
        for layer in layers[-n:]:
            for param in layer.parameters():
                param.requires_grad = True

    setFrozen(imageModel, True)
    layers = imageModel.model.transformer.layers
    freezeLayers(layers)

    for param in imageModel.head.parameters():
        param.requires_grad = True

    setFrozen(textModel, True)
    layers = textModel.model.encoder.layers
    freezeLayers(layers)

    for param in textModel.head.parameters():
        param.requires_grad = True

def getParamGroups(model, headLR, layerLR):
    """
    Split a model's trainable params into head vs transformer layer groups.
    """
    headParamIds = {id(p) for p in model.head.parameters()}

    headParams = [
        p for p in model.parameters()
        if p.requires_grad and id(p) in headParamIds
    ]
    layerParams = [
        p for p in model.parameters()
        if p.requires_grad and id(p) not in headParamIds
    ]

    return [
        {"params": headParams,  "lr": headLR},
        {"params": layerParams, "lr": layerLR},
    ]

def makeOptimizer(config, imageModel, textModel):
    layerLR = 0.1 * config.learningRate

    paramGroups = (
        getParamGroups(imageModel, config.learningRate, layerLR)
        + getParamGroups(textModel, config.learningRate, layerLR)
    )

    return torch.optim.Adam(paramGroups)

In [6]:
queryConfig = Config().load(os.path.join("configs", "querying.json"))

In [7]:
sharedDim = queryConfig.sharedDim

textModelName = "openai/clip-vit-base-patch32"
textModel = CLIPTextEmbedder(textModelName, sharedDim).to(device)

# textModelName = "bert-base-uncased"
# textModel = BERTTextEmbedder(textModelName, sharedDim).to(device)

vit, imageConfig = ViT.load(os.path.join("checkpoints", "pretrain", "latest"))
imageModel = ViTEmbedder(vit, sharedDim).to(device)

dataset = CombinedQueryData(imageConfig.dataset, training=True)
dataset.method = "none"

dataset.setTokenizer(textModelName)

experimentName = "ViT" + " " + textModelName.replace("/", "-")

imageModel, textModel = trainModel(queryConfig, textModel, imageModel, dataset, experimentName, datetime.now(), imageConfig)

saveExperiment(imageModel, textModel, imageConfig, experimentName, datetime.now())

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.pre_layrn


Loading MyFonts images from dataset ====================


Fonts serialized: 1855/3794google\fonts\ofl\kumarone\KumarOne-Regular.ttf execution context too long
Fonts serialized: 2335/3794google\fonts\ofl\notocoloremojicompattest\NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3735/3794google\fonts\ofl\zcoolxiaowei\ZCOOLXiaoWei-Regular.ttf [Errno 22] Invalid argument: 'google\\bitmaps\\????? ?? al.bmp'
Fonts serialized: 3794/3794


Fonts serialized: 36/18623dafont\fonts\135atom_sans.ttf [Errno 2] No such file or directory: 'dafont\\bitmaps\\13/5Atom Sans Regular al.bmp'
Fonts serialized: 79/18623dafont\fonts\50fifty.ttf [Errno 2] No such file or directory: 'dafont\\bitmaps\\50/fifty Regular al.bmp'
Fonts serialized: 353/18623dafont\fonts\adrenaline_zero.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Adrenaline : Zero Adrenaline : Zero al.bmp'
Fonts serialized: 410/18623dafont\fonts\aggstock.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\?ggstock Regula

cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.


Fonts serialized: 3910/18623

...

Fonts serialized: 7066/18623dafont\fonts\jerónimo_cartoon.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Jer?nimo cartoon Regular al.bmp'
Fonts serialized: 7162/18623dafont\fonts\julia_black_extended.ttf Not a TrueType or OpenType font (bad sfntVersion)
Fonts serialized: 7317/18623dafont\fonts\kaylafiz.ttf division by zero
Fonts serialized: 7979/18623dafont\fonts\lavoshandy_99.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\LavosHandy? Regular al.bmp'
Fonts serialized: 8089/18623dafont\fonts\lettif__.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Letters II "Fenotype" al.bmp'
Fonts serialized: 8161/18623dafont\fonts\like_giselle.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Like Giselle? Regular al.bmp'
Fonts serialized: 8285/18623dafont\fonts\lomonesia-_dingbats.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Lomonesia? Dingbats Regular al.bmp'
Fonts serialized: 8322/18623dafont\fonts\louis_george_cafe.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Louis George 

f:\PycharmProjects\Briefcase\utils\loaders\dafont.py:14: ParserWarning: Skipping line 1273: expected 6 fields, saw 7
Skipping line 2018: expected 6 fields, saw 7
Skipping line 2205: expected 6 fields, saw 8
Skipping line 3214: expected 6 fields, saw 7
Skipping line 7537: expected 6 fields, saw 7
Skipping line 7538: expected 6 fields, saw 7
Skipping line 7539: expected 6 fields, saw 7
Skipping line 7931: expected 6 fields, saw 8
Skipping line 7932: expected 6 fields, saw 8
Skipping line 8343: expected 6 fields, saw 8
Skipping line 9171: expected 6 fields, saw 7
Skipping line 9419: expected 6 fields, saw 8
Skipping line 9420: expected 6 fields, saw 7
Skipping line 9470: expected 6 fields, saw 8
Skipping line 9856: expected 6 fields, saw 8
Skipping line 9934: expected 6 fields, saw 7
Skipping line 10059: expected 6 fields, saw 7
Skipping line 10060: expected 6 fields, saw 7
Skipping line 10092: expected 6 fields, saw 8
Skipping line 10093: expected 6 fields, saw 8
Skipping line 10096: exp

2246015 33294 2246015
90.98% of fonts have descriptions


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\dylan\_netrc.
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


1 | 4551/51130 | 8.901% |  Train Loss: 9.02 | Test Loss: 8.44